# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanwajid09/Flyrank-intern/blob/main/work/notebooks/w07_action_playbook.ipynb)

**Lane:** Content Refresh Opportunity Scoring

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The final action queue combines the model’s probability with the baseline’s reason codes to produce a ranked list that a content editor can action immediately.

| Priority | Action | Reason Code | When to use |
|---|---|---|---|
| 1 | **Refresh** | `stale_visible_page` | Page has high visibility but hasn’t been updated in 180+ days. Highest ROI. |
| 2 | **Refresh** | `page_one_decay_risk` | Page ranks on page 1 but is aging. Risk of losing position. |
| 3 | **Expand & Refresh** | `thin_visible_page` | Page gets traffic but has thin content (<1200 words). Expand depth. |
| 4 | **Refresh Metadata** | `low_ctr_visible_page` | Page is visible but CTR is below expected. Title/description review. |
| 5 | **Monitor** | `general_review` | No specific signal fires. Add to watch list for next cycle. |

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import json, os

# Load and prepare
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].fillna(0)
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna('unknown')
df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])

# Train model on full data for final queue
num_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct'
]
cat_features = ['competition_level', 'content_type', 'main_intent', 'age_tier',
                'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

X_num = df[num_features].copy()
X_cat = df[cat_features].copy()
for col in X_cat.columns:
    le = LabelEncoder()
    X_cat[col] = le.fit_transform(X_cat[col].astype(str))
X = pd.concat([X_num, X_cat], axis=1)
y = df['is_declining_label']

rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X, y)
df['model_prob'] = rf.predict_proba(X)[:, 1]

# Reason codes
def get_reason(row):
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return 'stale_visible_page'
    if row['avg_position'] > 0 and row['avg_position'] <= 10 and row['content_age_days'] >= 180:
        return 'page_one_decay_risk'
    if row['word_count'] > 0 and row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        return 'thin_visible_page'
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        return 'low_ctr_visible_page'
    return 'general_review'

def get_action(reason):
    actions = {'thin_visible_page': 'expand_and_refresh', 'stale_visible_page': 'refresh',
               'page_one_decay_risk': 'refresh', 'low_ctr_visible_page': 'refresh_metadata'}
    return actions.get(reason, 'monitor')

df['reason_code'] = df.apply(get_reason, axis=1)
df['action_label'] = df['reason_code'].apply(get_action)
df['final_rank'] = df['model_prob'].rank(method='first', ascending=False).astype(int)

print('Action distribution:')
print(df['action_label'].value_counts().to_string())
print(f'\nPages with model_prob > 0.7: {(df["model_prob"] > 0.7).sum()}')
print(f'Pages with model_prob > 0.5: {(df["model_prob"] > 0.5).sum()}')

Action distribution:
[computed at runtime]


## 2. Intended use and limits

**Who uses this:** Content editors and SEO managers at FlyRank’s client organizations. The queue tells them which pages to review first each week.

**Intended use:** Decision-support tool for weekly content refresh prioritization. The editor reviews each flagged page, decides if a refresh is warranted based on their domain knowledge, and either refreshes or dismisses.

**Where it stops being valid:**
- ❌ Not valid for predicting Google algorithm changes
- ❌ Not valid for pages with fewer than 90 days of history (insufficient data)
- ❌ Not valid as an automated "refresh and forget" system — human review is mandatory
- ❌ Not valid for causal claims (“refreshing will restore traffic”)
- ❌ Not tested on the full warehouse release — results may differ at scale

In [2]:
print('=== Intended Use Summary ===')
print('Target user: Content editors / SEO managers')
print('Use case: Weekly refresh prioritization (decision-support)')
print('NOT valid for: Causal claims, algorithm prediction, automated actions')
print(f'\nCoverage: {len(df)} pages scored')
print(f'Actionable (prob > 0.5): {(df["model_prob"] > 0.5).sum()} pages')
print(f'High confidence (prob > 0.7): {(df["model_prob"] > 0.7).sum()} pages')

=== Intended Use Summary ===
Target user: Content editors / SEO managers
Use case: Weekly refresh prioritization (decision-support)
NOT valid for: Causal claims, algorithm prediction, automated actions


## 3. Human review + the no-go list

**What a person must check before acting:**
1. Is the page actually outdated, or is it intentionally evergreen?
2. Is the decline real or seasonal (check year-over-year if available)?
3. Does the page serve a legal/compliance purpose that shouldn’t be modified?
4. Are there external factors (site migration, URL change) causing the signal?

**The no-go list (never automate these):**
- ❌ Never auto-delete a page based on model output alone
- ❌ Never auto-publish refreshed content without human review
- ❌ Never use this queue to justify client billing decisions
- ❌ Never treat a low model score as proof that a page is healthy

In [3]:
print('=== Human Review Checklist ===')
print('Before acting on any recommendation:')
print('  1. Is the page actually outdated or intentionally evergreen?')
print('  2. Is the decline real or seasonal?')
print('  3. Does the page serve legal/compliance purposes?')
print('  4. Are there external factors (migration, URL changes)?')
print()
print('NEVER automate:')
print('  - Page deletion')
print('  - Content publishing without review')
print('  - Client billing decisions')
print('  - Treating low score as proof of health')

=== Human Review Checklist ===
Before acting on any recommendation:
  1. Is the page actually outdated or intentionally evergreen?
  2. Is the decline real or seasonal?
  3. Does the page serve legal/compliance purposes?
  4. Are there external factors (migration, URL changes)?


## 4. Monitoring / retrain triggers

**The queue goes stale when:**
1. **Data drift:** The distribution of impressions/positions shifts significantly (e.g., >20% change in mean)
2. **Label drift:** The base declining rate changes by >5 percentage points
3. **New content types:** FlyRank adds a new content type not seen in training
4. **Time elapsed:** >90 days since last model training

**Retrain trigger checklist:**
- [ ] Base declining rate has changed by >5pp
- [ ] >10% of pages have a new content type
- [ ] Mean impressions shifted by >20%
- [ ] >90 days since last training run

In [4]:
print('=== Monitoring Triggers ===')
print(f'Current base declining rate: {df["is_declining_label"].mean():.3f}')
print(f'Current mean impressions: {df["impressions_90d"].mean():.0f}')
print(f'Current content type distribution:')
print(df['content_type'].value_counts(normalize=True).round(3).to_string())
print()
print('Retrain when any of these change significantly.')

=== Monitoring Triggers ===
[computed at runtime]


## 5. Exports for the paper

Write the final action queue and feature importance to `work/outputs/` for the capstone paper.

In [5]:
# Export final queue
os.makedirs('../../work/outputs', exist_ok=True)
out_cols = ['content_id', 'client_id', 'final_rank', 'model_prob',
            'reason_code', 'action_label', 'is_declining_label',
            'impressions_90d', 'avg_position', 'days_since_last_update']
final_queue = df[out_cols].sort_values('final_rank')
final_queue.to_csv('../../work/outputs/final_action_queue.csv', index=False)

# Export feature importance
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
feat_imp.to_json('../../work/outputs/feature_importance.json')

print(f'Final action queue: {len(final_queue)} rows written to work/outputs/final_action_queue.csv')
print(f'Feature importance: written to work/outputs/feature_importance.json')
print(f'\nTop 5 actions in queue:')
print(final_queue.head(5)[['final_rank', 'model_prob', 'reason_code', 'action_label']].to_string(index=False))

Final action queue written to work/outputs/


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.